In [1]:
from ase.io import read
from ase.atoms import Atoms
from itertools import chain
import json
from pathlib import Path
from multiprocessing import Pool
from pymongo import MongoClient

from colabfit.tools.configuration import AtomicConfiguration
from colabfit.tools.property_definitions import potential_energy_pd, free_energy_pd
from time import time
import zipfile
import numpy as np
from math import isnan

from colabfit.tools.configuration import AtomicConfiguration
from colabfit.tools.database import generate_ds_id, load_data, MongoDatabase

In [4]:
"MgTa2N3 A reference Dirac semimetal".lower().replace(" ", "_")

'mgta2n3_a_reference_dirac_semimetal'

In [ ]:
# cl = MongoClient(f"mongodb://localhost:27017")
# db = cl["mem_test"]
# res = db.command("profile", 2)

In [2]:
client = MongoDatabase(
    database_name="cf-update-2023-11-30", uri="mongodb://10.32.250.13:30007"
)

In [11]:
def build_do(do_result, client_name, client_uri):
    co_id = do_result["relationships"][0]["configuration"]
    pi_ids = do_result["relationships"][0]["property_instance"]
    client = MongoDatabase(client_name, uri=client_uri)
    co = client.configurations.find_one({"colabfit-id": co_id})
    pis = list(client.property_instances.find({"colabfit-id": {"$in": pi_ids}}))
    a = Atoms(
        numbers=co["atomic_numbers"],
        positions=co["positions"],
        cell=co["cell"],
        pbc=co["pbc"],
    )
    a.info["name"] = ", ".join([x.strip() for x in co["names"]])
    for i, pi in enumerate(pis):
        # print ('pi_type',pi['type'])
        for k, v in pi.items():
            if isinstance(v, dict):
                for k2, v2 in v.items():
                    if isinstance(v2, dict):
                        if "source-value" in pi[k][k2]:
                            # hardcode whether property goes in info or arrays for now
                            if k2 == "forces":
                                a.arrays["forces"] = np.array(pi[k][k2]["source-value"])
                                a.info["forces-units"] = pi[k][k2]["source-unit"]
                            elif k2 == "stress":
                                # Needed so that ASE formats properly
                                a.info["stress"] = np.array(pi[k][k2]["source-value"])
                                a.info["stress-units"] = pi[k][k2]["source-unit"]
                            else:
                                a.info["%s.%s" % (k, k2)] = pi[k][k2]["source-value"]
                                units = pi[k][k2].get("source-unit")
                                if units:
                                    a.info["%s.source-unit" % (k)] = units

    return a

In [3]:
SHORT_ID_STRING_NAME = "colabfit-id"
import multiprocessing
from tqdm import tqdm
from functools import partial
from ase.io import write as ase_write
from itertools import chain

In [ ]:
def export_ds_to_xyz(self, ds_id, nprocs=1):
    ds_doc = self.datasets.find_one({SHORT_ID_STRING_NAME: {"$eq": ds_id}})
    do_results = self.data_objects.find(
        {"relationships.dataset": ds_id},
        {"colabfit-id": 1, "relationships": 1},
    )
    p = multiprocessing.Pool(nprocs)
    part_build_do = partial(
        build_do, client_name=self.database_name, client_uri=self.uri
    )
    results = itertools.chain.from_iterable(p.imap(part_build_do, do_results))
    p.close()
    p.join()
    ase_write("%s.xyz" % ds_doc["extended-id"], results)

In [6]:
do_results = client.data_objects.find(
    {"relationships.dataset": "DS_b4o9n4n8b7sp_0"},
    {"colabfit-id": 1, "relationships": 1},
)

In [12]:
build_do(
    do_result=next(do_results),
    client_name="cf-update-2023-11-30",
    client_uri="mongodb://10.32.250.13:30007",
)

potential-energy energy {'source-value': -542.86554042, 'source-unit': 'eV'}
{'name': 'Zn__row1__2', 'forces-units': 'eV/angstrom', 'potential-energy.energy': -542.86554042, 'potential-energy.source-unit': 'eV'}
potential-energy per-atom {'source-value': False}
{'name': 'Zn__row1__2', 'forces-units': 'eV/angstrom', 'potential-energy.energy': -542.86554042, 'potential-energy.source-unit': 'eV', 'potential-energy.per-atom': False}


Atoms(symbols='Mg382Zn', pbc=False, cell=[[24.6019382477, -42.6118070097, 0.0], [-49.2038764954, 0.0, 0.0], [0.0, 0.0, -5.1909956932]], forces=...)

In [ ]:
def export_ds_to_xyz(self, ds_id, nprocs=1):
    ds_doc = self.datasets.find_one({SHORT_ID_STRING_NAME: {"$eq": ds_id}})
    do_results = self.data_objects.find(
        {"relationships.dataset": ds_id},
        {"colabfit-id": 1, "relationships": 1},
    )
    p = multiprocessing.Pool(nprocs)
    part_build_do = partial(
        build_do, client_name=self.database_name, client_uri=self.uri
    )
    # results = p.imap(part_build_do, do_results)
    results = [part_build_do(do_result) for do_result in do_results]
    # p.close()
    # p.join()
    ase_write("%s.xyz" % ds_doc["extended-id"], results)
    return results

In [3]:
client.export_ds_to_xyz("DS_b4o9n4n8b7sp_0", nprocs=8)

In [ ]:
res2 = cl.system.profile.find({})

In [ ]:
res2_list = [x for x in res2]

In [ ]:
cd = {
    "type": "float",
    "has-unit": True,
    "extent": {"a": ["b", "a"]},
    "required": False,
    "description": "If known, represents values which have been subtracted from the base energy, resulting in the formation energy.",
}
dc = {
    "type": "float",
    "has-unit": True,
    "extent": {"a": ["a", "b"]},
    "required": False,
    "description": "If known, represents values which have been subtracted from the base energy, resulting in the formation energy.",
}